# SGDRegressor — Cars93 (MPG)

Objectif : prédire la consommation **MPG.highway** à partir des caractéristiques des voitures.



In [3]:
# Installation (utile hors Colab)
# Sur Google Colab, ces libs sont deja disponibles
# Si besoin, decommenter :
!pip -q install -r requirements.txt


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.46.0 requires packaging<26,>=20, but you have packaging 26.0 which is incompatible.


## 1) Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import SGDRegressor

pd.options.display.max_columns = None

## 2) Paramètres à explorer

**Consigne** : modifiez ces valeurs, relancez l'entraînement et observez l'effet sur les courbes.

- `eta0` : learning rate (taux d'apprentissage)
- `batch_size` : taille des mini-batchs (entraînement par lots)
- `alpha` : régularisation L2 (plus grand => plus de pénalisation)

In [ ]:
CONFIG = {
    "random_state": 42,
    "test_size": 0.2,
    "epochs": 40,
    "batch_size": 16,
    "eta0": 0.01,
    "alpha": 0.0001,
}

## 3) Chargement des données

Dataset : **Cars93** (source CSV publique). On prédit **MPG.highway**.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/SkatAI/efrei-st2mml/refs/heads/master/data/Cars93.csv"


def load_data(url: str) -> pd.DataFrame:
    df = pd.read_csv(url)
    return df


df = load_data(DATA_URL)
df.head()

## 4) Préparation des features

- On retire la cible `MPG.highway` (et `MPG.city` pour éviter toute fuite)
- Encodage **one-hot** des variables catégorielles
- Normalisation des variables numériques

In [ ]:
TARGET = "MPG.highway"
DROP_COLS = ["MPG.highway", "MPG.city"]


def split_features_target(df: pd.DataFrame):
    X = df.drop(columns=DROP_COLS)
    y = df[TARGET].astype(float).values
    return X, y


def build_preprocessor(X: pd.DataFrame):
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ],
        remainder="drop",
    )
    return preprocessor, num_cols, cat_cols


X, y = split_features_target(df)
preprocessor, num_cols, cat_cols = build_preprocessor(X)

print("Numeriques:", num_cols)
print("Categoriels:", cat_cols)

## 5) Split train / test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CONFIG["test_size"], random_state=CONFIG["random_state"]
)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

## 6) Entraînement mini-batch (SGD)

On entraîne **manuellement** par mini-batchs avec `partial_fit` pour pouvoir
contrôler le **batch size** et suivre l'évolution des performances par époque.

In [ ]:
def train_sgd_minibatch(X_train, y_train, X_val, y_val, config):
    model = SGDRegressor(
        loss="squared_error",
        penalty="l2",
        alpha=config["alpha"],
        learning_rate="constant",
        eta0=config["eta0"],
        random_state=config["random_state"],
    )

    rng = np.random.default_rng(config["random_state"])
    n = X_train.shape[0]
    history = []

    for epoch in range(config["epochs"]):
        indices = rng.permutation(n)
        for start in range(0, n, config["batch_size"]):
            batch_idx = indices[start : start + config["batch_size"]]
            X_batch = X_train[batch_idx]
            y_batch = y_train[batch_idx]
            model.partial_fit(X_batch, y_batch)

        y_pred = model.predict(X_val)
        rmse = mean_squared_error(y_val, y_pred, squared=False)
        history.append(rmse)

    return model, history

## 7) Évaluation et visualisations

In [ ]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    mae = mean_absolute_error(y_test, y_pred)
    return rmse, mae, y_pred


def plot_history(history):
    plt.figure(figsize=(7, 4))
    plt.plot(history, marker="o")
    plt.title("RMSE par époque")
    plt.xlabel("Époque")
    plt.ylabel("RMSE")
    plt.grid(True)
    plt.show()


def plot_predictions(y_true, y_pred):
    plt.figure(figsize=(5, 5))
    plt.scatter(y_true, y_pred, alpha=0.7)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], "r--")
    plt.xlabel("MPG.highway (réel)")
    plt.ylabel("MPG.highway (prédit)")
    plt.title("Prédictions vs Réel")
    plt.grid(True)
    plt.show()

## 8) Entraîner + évaluer

In [ ]:
model, history = train_sgd_minibatch(X_train, y_train, X_test, y_test, CONFIG)
rmse, mae, y_pred = evaluate_model(model, X_test, y_test)

print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")

plot_history(history)
plot_predictions(y_test, y_pred)

## 9) Expérimenter les paramètres

**Exercices** :
1. Essayez `eta0 = 0.1`, puis `eta0 = 0.001` (que se passe-t-il ?)
2. Essayez `batch_size = 1` vs `batch_size = 64`
3. Essayez `alpha = 0.0`, `alpha = 0.001`, `alpha = 0.01`

Pour comparer rapidement, vous pouvez tester plusieurs configurations.

In [ ]:
EXPERIMENTS = [
    {"name": "eta0=0.1", "eta0": 0.1, "batch_size": 16, "alpha": 0.0001},
    {"name": "eta0=0.001", "eta0": 0.001, "batch_size": 16, "alpha": 0.0001},
    {"name": "batch=1", "eta0": 0.01, "batch_size": 1, "alpha": 0.0001},
    {"name": "batch=64", "eta0": 0.01, "batch_size": 64, "alpha": 0.0001},
    {"name": "alpha=0.0", "eta0": 0.01, "batch_size": 16, "alpha": 0.0},
    {"name": "alpha=0.01", "eta0": 0.01, "batch_size": 16, "alpha": 0.01},
]

results = []

for exp in EXPERIMENTS:
    cfg = {**CONFIG, **exp}
    model, history = train_sgd_minibatch(X_train, y_train, X_test, y_test, cfg)
    rmse, mae, _ = evaluate_model(model, X_test, y_test)
    results.append((exp["name"], rmse, mae))

pd.DataFrame(results, columns=["config", "rmse", "mae"]).sort_values("rmse")